In [2]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

In [12]:
base_path = Path('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data')
belief_col = sorted([
  d.name for d in base_path.iterdir()
  if d.is_dir() and d.name.startswith('Belief_')
])
len(belief_col)

28

In [5]:
strat_frame = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/state_LP_cleaned_sf.csv")
df = pd.read_pickle('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/preprocessed_cleaned_all_result.pkl')

predictor_cols = [
    "Demo_sex_Symbol",
    "Demo_age_Symbol",
    "Demo_ethnicity_Symbol",
    "Demo_income_Symbol",
    "Demo_education_Symbol",
    "Demo_past_vote_Symbol",
    "time_window_start",
    "state_cleaned"
]
df = df.dropna(subset=belief_col + predictor_cols)

# find all time windows
df['time_window_start'] = pd.to_datetime(df['time_window_start'])
time_window_sorted = df['time_window_start'].unique()
time_window_sorted = sorted(time_window_sorted)
time_window_map = {window: idx + 1 for idx, window in enumerate(time_window_sorted)}
df.loc[:, 'time_window_start_idx'] = df['time_window_start'].map(time_window_map)
time_window_mapping = df[['time_window_start', 'time_window_start_idx']].drop_duplicates().sort_values('time_window_start_idx')

In [8]:
for belief in belief_col:
    base_dir = f"/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/{belief}"
    # all_national_summaries = []

    for window in time_window_mapping['time_window_start']:
        window_label = pd.Timestamp(window).strftime('%Y-%m-%d')
        model_dir = f"{base_dir}/{belief}_{window_label}_stan_output"
        national_done_path = f"{model_dir}/national_predictions.csv"
        if os.path.exists(national_done_path):
            print(f"[{window_label}] National already exists, skipping")
            # still load the summary for the timeseries
            # existing = pd.read_csv(national_done_path)
            # all_national_summaries.append(existing.iloc[0].to_dict())
            continue
        state_draws_path = f"{model_dir}/group_state_all_draws.csv"
        if not os.path.exists(state_draws_path):
            print(f"Missing: {state_draws_path}, skipping window {window_label}")
            continue

        state_draws = pd.read_csv(state_draws_path)
        draw_cols = [c for c in state_draws.columns if c.startswith('draw_')]

        # get state population weights from strat_frame
        # N_star.median is the expected population count per cell
        state_weights = (
            strat_frame
            .groupby('state_simple')['N_star.median']
            .sum()
            .reset_index()
            .rename(columns={'N_star.median': 'state_pop'})
        )

        merged = state_draws.merge(state_weights, on='state_simple', how='left')

        # national = population-weighted mean across states, per draw
        w = merged['state_pop'].values
        draw_matrix = merged[draw_cols].values  # (n_states, n_draws)
        national_draws_vals = (draw_matrix * w[:, None]).sum(axis=0) / w.sum()  # (n_draws,)

        national_summary = {
            "time_window_start": window,
            "pred_mean":     national_draws_vals.mean(),
            "pred_median":   np.median(national_draws_vals),
            "pred_lower_95": np.percentile(national_draws_vals, 2.5),
            "pred_upper_95": np.percentile(national_draws_vals, 97.5),
            "pred_lower_90": np.percentile(national_draws_vals, 5),
            "pred_upper_90": np.percentile(national_draws_vals, 95),
            "pred_lower_80": np.percentile(national_draws_vals, 10),
            "pred_upper_80": np.percentile(national_draws_vals, 90),
            "pred_lower_68": np.percentile(national_draws_vals, 16),
            "pred_upper_68": np.percentile(national_draws_vals, 84),
            "pred_sd":       national_draws_vals.std()
        }
        # all_national_summaries.append(national_summary)

        # also save per-window file to match original structure
        national_draws_df = pd.DataFrame({
            "draw": draw_cols,
            "expected_value": national_draws_vals,
            "weight_col": "state_pop_median"
        })
        national_draws_df["time_window_start"] = window
        national_draws_df.to_csv(f"{model_dir}/national_all_draws.csv", index=False)

        summary_df = pd.DataFrame([national_summary])
        summary_df.to_csv(f"{model_dir}/national_predictions.csv", index=False)

        print(f"  [{window_label}] national mean={national_summary['pred_mean']:.4f}")

    # # save full timeseries
    # if all_national_summaries:
    #     ts_df = pd.DataFrame(all_national_summaries).sort_values('time_window_start')
    #     ts_df.to_csv(f"{base_dir}/national_predictions_timeseries.csv", index=False)
    #     print(f"Saved national timeseries for: {belief}")

[2023-11-05] National already exists, skipping
[2023-11-19] National already exists, skipping
[2023-12-03] National already exists, skipping
[2023-12-17] National already exists, skipping
[2023-12-31] National already exists, skipping
[2024-01-14] National already exists, skipping
[2024-01-28] National already exists, skipping
[2024-02-11] National already exists, skipping
[2024-02-25] National already exists, skipping
[2024-03-10] National already exists, skipping
[2024-03-24] National already exists, skipping
[2024-04-07] National already exists, skipping
[2024-04-21] National already exists, skipping
[2024-05-05] National already exists, skipping
[2024-05-19] National already exists, skipping
[2024-06-02] National already exists, skipping
[2024-06-16] National already exists, skipping
[2024-06-30] National already exists, skipping
[2024-07-14] National already exists, skipping
[2024-07-28] National already exists, skipping
[2024-08-11] National already exists, skipping
[2024-08-25] 